# Get Raw NRN Data

## Purpose

This notebook downloads and organizes the National Road Network (NRN) GeoPackage files required by the Geospatial-CANOE preprocessing workflow.

No road filtering, spatial transformation, graph construction, or CANOE schema encoding is performed here. The goal is only to acquire the raw NRN files and place them in the expected project directory structure.

---

## Data Source

**National Road Network / Réseau routier national**

- Source: Government of Canada Open Data
- Format: GeoPackage (`.gpkg`) distributed as ZIP archives
- Coverage: Provincial and territorial road networks
- Language used: English (`*_en.gpkg`)
- Purpose: Raw road-network input for later construction of national backbone and freight-access transport layers.

Source page:

```text
https://open.canada.ca/data/en/dataset/3d282116-e556-400c-9306-ca1a3cada77f
```

---

## Expected Output Structure

```text
data_files/
└── raw/
    └── nrn/
        ├── AB/
        ├── BC/
        ├── MB/
        ├── NB/
        ├── NL/
        ├── NS/
        ├── NT/
        ├── NU/
        ├── ON/
        ├── PE/
        ├── QC/
        ├── SK/
        └── YT/
```

Each province or territory folder should contain one English NRN GeoPackage:

```text
*_en.gpkg
```

---

## Workflow

This notebook performs the following stages:

1. Create the required NRN raw-data folders.
2. Define the provincial and territorial NRN download URLs.
3. Download ZIP archives sequentially.
4. Extract each archive into its province or territory folder.
5. Validate that each folder contains exactly one English GeoPackage.

Later scripts assume this raw NRN directory structure already exists.

In [1]:
# =============================================================================
# Cell 2 — Imports and project paths
# =============================================================================

from pathlib import Path
import shutil
import zipfile
import time

import requests


# Project root
# Assumes this notebook is located in:
#   notebooks/acquire_raw_nrn_data.ipynb
PROJECT_ROOT = Path.cwd().parents[0]

# If running from a different working directory, manually set:
# PROJECT_ROOT = Path(r"C:\Users\aviga\Research\repos\temoa_geospace")


# Raw NRN data directory
DATA_FILES = PROJECT_ROOT / "data_files"
RAW_DATA = DATA_FILES / "raw"
RAW_NRN = RAW_DATA / "nrn"


# Province and territory codes used by the NRN workflow
PROVINCES = [
    "AB", "BC", "MB", "NB", "NL", "NS",
    "NT", "NU", "ON", "PE", "QC", "SK", "YT",
]


# Create required NRN directories
RAW_NRN.mkdir(parents=True, exist_ok=True)

for province in PROVINCES:
    (RAW_NRN / province).mkdir(parents=True, exist_ok=True)


print(f"Project root: {PROJECT_ROOT}")
print(f"Raw NRN folder: {RAW_NRN}")

Project root: c:\Users\aviga\Research\repos\temoa_geospace
Raw NRN folder: c:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\nrn


In [2]:
# =============================================================================
# Cell 3 — Download settings
# =============================================================================

# HTTP request settings
HEADERS = {
    "User-Agent": (
        "Geospatial-CANOE/0.1.0 "
        "(University of Toronto; Andrew Vigars)"
    )
}

REQUEST_TIMEOUT = 120          # seconds
MAX_RETRIES = 3
DOWNLOAD_DELAY = 2             # seconds between downloads
CHUNK_SIZE = 1024 * 1024       # 1 MB streaming chunks

print("Download settings configured.")

Download settings configured.


In [3]:
# =============================================================================
# Cell 4 — NRN source definition
# =============================================================================
# National Road Network / Réseau routier national
#
# Source page:
# https://open.canada.ca/data/en/dataset/3d282116-e556-400c-9306-ca1a3cada77f
#
# The NRN GeoPackage downloads follow a stable URL pattern:
# https://geo.statcan.gc.ca/nrn_rrn/{province}/nrn_rrn_{province}_GPKG.zip
#
# where {province} is the lowercase province or territory code.
#
# This cell only defines the NRN source. It does not download anything.

NRN_SOURCE_PAGE = (
    "https://open.canada.ca/data/en/dataset/"
    "3d282116-e556-400c-9306-ca1a3cada77f"
)

NRN_BASE_URL = "https://geo.statcan.gc.ca/nrn_rrn"

NRN_PROVINCE_CODES = {
    "AB": "ab",
    "BC": "bc",
    "MB": "mb",
    "NB": "nb",
    "NL": "nl",
    "NS": "ns",
    "NT": "nt",
    "NU": "nu",
    "ON": "on",
    "PE": "pe",
    "QC": "qc",
    "SK": "sk",
    "YT": "yt",
}

NRN_RESOURCES = {
    province: {
        "url": f"{NRN_BASE_URL}/{code}/nrn_rrn_{code}_GPKG.zip",
        "output_dir": RAW_NRN / province,
        "archive_name": f"nrn_rrn_{code}_GPKG.zip",
        "expected_pattern": "*_en.gpkg",
    }
    for province, code in NRN_PROVINCE_CODES.items()
}

print("NRN source configured")
print(f"Source page: {NRN_SOURCE_PAGE}")
print(f"Number of provincial/territorial resources: {len(NRN_RESOURCES)}")

for province, resource in NRN_RESOURCES.items():
    print(f"{province}: {resource['url']}")

NRN source configured
Source page: https://open.canada.ca/data/en/dataset/3d282116-e556-400c-9306-ca1a3cada77f
Number of provincial/territorial resources: 13
AB: https://geo.statcan.gc.ca/nrn_rrn/ab/nrn_rrn_ab_GPKG.zip
BC: https://geo.statcan.gc.ca/nrn_rrn/bc/nrn_rrn_bc_GPKG.zip
MB: https://geo.statcan.gc.ca/nrn_rrn/mb/nrn_rrn_mb_GPKG.zip
NB: https://geo.statcan.gc.ca/nrn_rrn/nb/nrn_rrn_nb_GPKG.zip
NL: https://geo.statcan.gc.ca/nrn_rrn/nl/nrn_rrn_nl_GPKG.zip
NS: https://geo.statcan.gc.ca/nrn_rrn/ns/nrn_rrn_ns_GPKG.zip
NT: https://geo.statcan.gc.ca/nrn_rrn/nt/nrn_rrn_nt_GPKG.zip
NU: https://geo.statcan.gc.ca/nrn_rrn/nu/nrn_rrn_nu_GPKG.zip
ON: https://geo.statcan.gc.ca/nrn_rrn/on/nrn_rrn_on_GPKG.zip
PE: https://geo.statcan.gc.ca/nrn_rrn/pe/nrn_rrn_pe_GPKG.zip
QC: https://geo.statcan.gc.ca/nrn_rrn/qc/nrn_rrn_qc_GPKG.zip
SK: https://geo.statcan.gc.ca/nrn_rrn/sk/nrn_rrn_sk_GPKG.zip
YT: https://geo.statcan.gc.ca/nrn_rrn/yt/nrn_rrn_yt_GPKG.zip


In [4]:
# =============================================================================
# Cell 5 — Download and extraction helper functions
# =============================================================================

def download_file(url, destination):
    """
    Download a file from a URL if it does not already exist.
    """

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        print(f"[Skip download] {destination.name} already exists.")
        return destination

    print(f"[Download] {destination.name}")

    response = requests.get(
        url,
        headers=HEADERS,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:
                f.write(chunk)

    print(f"[Complete download] {destination.name}")

    time.sleep(DOWNLOAD_DELAY)

    return destination


def extract_flatten_and_clean(zip_path, output_dir, overwrite=False):
    """
    Extract an NRN ZIP archive, flatten any GeoPackage files into the
    province/territory folder, validate English/French files, then delete
    the ZIP archive and nested extracted folder(s).
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    existing_gpkgs = sorted(output_dir.glob("*.gpkg"))

    if existing_gpkgs and not overwrite:
        print(f"[Skip extract] {output_dir.name} already has GeoPackage file(s).")
        return existing_gpkgs

    print(f"[Extract] {zip_path.name} → {output_dir.name}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)

    all_gpkgs = sorted(output_dir.rglob("*.gpkg"))

    if not all_gpkgs:
        raise FileNotFoundError(
            f"No GeoPackage files found after extracting {zip_path.name}"
        )

    moved_gpkgs = []

    for source_path in all_gpkgs:
        destination_path = output_dir / source_path.name

        if source_path.parent == output_dir:
            moved_gpkgs.append(source_path)
            continue

        if destination_path.exists():
            if overwrite:
                destination_path.unlink()
            else:
                print(f"[Skip move] {destination_path.name} already exists.")
                moved_gpkgs.append(destination_path)
                continue

        shutil.move(str(source_path), str(destination_path))
        moved_gpkgs.append(destination_path)
        print(f"[Move] {source_path.name} → {destination_path.name}")

    for path in sorted(output_dir.iterdir()):
        if path.is_dir():
            shutil.rmtree(path)

    province = output_dir.name

    final_gpkgs = sorted(output_dir.glob("*.gpkg"))
    english_gpkgs = sorted(output_dir.glob(f"NRN_{province}_*_GPKG_en.gpkg"))
    french_gpkgs = sorted(output_dir.glob(f"RRN_{province}_*_GPKG_fr.gpkg"))

    if len(english_gpkgs) != 1:
        raise ValueError(
            f"{province}: expected 1 English NRN GPKG, found {len(english_gpkgs)}"
        )

    if len(french_gpkgs) != 1:
        raise ValueError(
            f"{province}: expected 1 French RRN GPKG, found {len(french_gpkgs)}"
        )

    if zip_path.exists():
        zip_path.unlink()
        print(f"[Delete] {zip_path.name}")

    print(f"[Complete] {province}: {len(final_gpkgs)} GeoPackage file(s)")

    return final_gpkgs

In [5]:
# =============================================================================
# Cell 6 — Download, extract, flatten, and clean NRN archives
# =============================================================================

acquired_files = {}

print("Acquiring National Road Network GeoPackages...\n")

for province, resource in NRN_RESOURCES.items():

    output_dir = resource["output_dir"]
    archive_path = output_dir / resource["archive_name"]

    downloaded_archive = download_file(
        url=resource["url"],
        destination=archive_path,
    )

    acquired_files[province] = extract_flatten_and_clean(
        zip_path=downloaded_archive,
        output_dir=output_dir,
        overwrite=False,
    )

print("\nNRN acquisition summary")
print("-----------------------")

for province, files in acquired_files.items():
    print(f"{province}: {len(files)} GeoPackage file(s)")
    for file in files:
        print(f"  - {file.name}")

print(f"\nProcessed {len(acquired_files)} provincial/territorial archive(s).")

Acquiring National Road Network GeoPackages...

[Download] nrn_rrn_ab_GPKG.zip
[Complete download] nrn_rrn_ab_GPKG.zip
[Extract] nrn_rrn_ab_GPKG.zip → AB
[Move] NRN_AB_17_0_GPKG_en.gpkg → NRN_AB_17_0_GPKG_en.gpkg
[Move] RRN_AB_17_0_GPKG_fr.gpkg → RRN_AB_17_0_GPKG_fr.gpkg
[Delete] nrn_rrn_ab_GPKG.zip
[Complete] AB: 2 GeoPackage file(s)
[Download] nrn_rrn_bc_GPKG.zip
[Complete download] nrn_rrn_bc_GPKG.zip
[Extract] nrn_rrn_bc_GPKG.zip → BC
[Delete] nrn_rrn_bc_GPKG.zip
[Complete] BC: 2 GeoPackage file(s)
[Download] nrn_rrn_mb_GPKG.zip
[Complete download] nrn_rrn_mb_GPKG.zip
[Extract] nrn_rrn_mb_GPKG.zip → MB
[Delete] nrn_rrn_mb_GPKG.zip
[Complete] MB: 2 GeoPackage file(s)
[Download] nrn_rrn_nb_GPKG.zip
[Complete download] nrn_rrn_nb_GPKG.zip
[Extract] nrn_rrn_nb_GPKG.zip → NB
[Move] NRN_NB_15_0_GPKG_en.gpkg → NRN_NB_15_0_GPKG_en.gpkg
[Move] RRN_NB_15_0_GPKG_fr.gpkg → RRN_NB_15_0_GPKG_fr.gpkg
[Delete] nrn_rrn_nb_GPKG.zip
[Complete] NB: 2 GeoPackage file(s)
[Download] nrn_rrn_nl_GPKG.zip
[